# Unidade 3 — Aula 3
## Criação de novos atributos
Ciência de Dados · Bacharelado em Engenharia de Software · UNIFSA


**Situação:** o dono de um restaurante quer saber quais clientes dão as melhores gorjetas e como organizar o atendimento. As colunas originais não respondem diretamente a essas perguntas. Nesta atividade, você vai criar novos atributos a partir das colunas existentes.

**Base de dados:** *Tips* (BRYANT, P. G.; SMITH, M. A., 1995). 244 contas registradas por um garçom em um restaurante dos Estados Unidos, com valores em dólares. A base é carregada de uma cópia pública:
https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv


## 1. Carga e preparação da base

In [2]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df = pd.read_csv(URL)
df.columns = ["conta", "gorjeta", "sexo", "fumante", "dia", "turno", "pessoas"]
df = df.drop_duplicates().reset_index(drop=True)
df.head()

,conta,gorjeta,sexo,fumante,dia,turno,pessoas
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


| Coluna | Descrição |
|---|---|
| conta | valor da conta, em dólares, sem a gorjeta |
| gorjeta | valor da gorjeta, em dólares |
| sexo | sexo de quem pagou a conta |
| fumante | se havia fumantes na mesa (Yes, No) |
| dia | dia da semana (Thur, Fri, Sat, Sun) |
| turno | almoço (Lunch) ou jantar (Dinner) |
| pessoas | número de pessoas na mesa |

## 2. Razão: dividir uma coluna por outra
**Exemplo:** percentual da gorjeta em relação à conta, e sua média por tamanho de grupo.

In [3]:
df["percentual_gorjeta"] = df["gorjeta"] / df["conta"] * 100
df.groupby("pessoas")[["gorjeta", "percentual_gorjeta"]].mean().round(2)

,gorjeta,percentual_gorjeta
pessoas,,
1,1.44,21.73
2,2.59,16.58
3,3.39,15.22
4,4.14,14.59
5,4.03,14.15
6,5.22,15.62


**Sua vez 2.1:** crie o atributo `conta_por_pessoa` (valor da conta dividido pelo número de pessoas) e calcule sua média por turno.

In [4]:
df["conta_por_pessoa"]=df["conta"]/df["pessoas"]
df.groupby("turno")[["conta","conta_por_pessoa"]].mean().round(2)

,conta,conta_por_pessoa
turno,,
Dinner,20.80,8.11
Lunch,17.23,7.33


**Pergunta 2.2:** em qual turno cada cliente gasta mais, individualmente? Que diferença existe entre olhar `conta` e olhar `conta_por_pessoa`?

**Resposta:**

Na janta, o cliente gasta mais individualmente. Conta é o somatório por mesa, e conta_por_pessoa é a conta total dividido pela quantidade de pessoas.

## 3. Soma: combinar colunas na mesma unidade
**Exemplo:** total efetivamente pago pelo cliente.

In [7]:
df["total_pago"] = df["conta"] + df["gorjeta"]
df[["conta", "gorjeta", "total_pago"]].head()

,conta,gorjeta,total_pago
0,16.99,1.01,18.00
1,10.34,1.66,12.00
2,21.01,3.50,24.51
3,23.68,3.31,26.99
4,24.59,3.61,28.20


**Sua vez 3.1:** crie o atributo `total_por_pessoa` (total pago dividido pelo número de pessoas) e mostre as cinco mesas com maior valor.

In [5]:
df["total_por_pessoa"] = (df["conta"] + df["gorjeta"])/df["pessoas"]
df[["conta", "gorjeta", "total_por_pessoa"]].head()

,conta,gorjeta,total_por_pessoa
0,16.99,1.01,9.000
1,10.34,1.66,4.000
2,21.01,3.50,8.170
3,23.68,3.31,13.495
4,24.59,3.61,7.050


## 4. Indicador: transformar uma condição em 0 ou 1
**Exemplo:** a mesa foi atendida no fim de semana? Comparação do percentual médio da gorjeta.

In [6]:
df["fim_de_semana"] = df["dia"].isin(["Sat", "Sun"]).astype(int)
df.groupby("fim_de_semana")["percentual_gorjeta"].agg(["count", "mean"]).round(2)

,count,mean
fim_de_semana,,
0,80,16.34
1,163,15.96


**Sua vez 4.1:** crie o indicador `grupo_grande`, igual a 1 para mesas com 4 pessoas ou mais e 0 nos demais casos. Compare o percentual médio da gorjeta entre os dois grupos.

In [11]:
df["grupo_grande"] = (df["pessoas"]>4).astype(int)
df.groupby("grupo_grande")["percentual_gorjeta"].agg(["count", "mean"]).round(2)

,count,mean
grupo_grande,,
0,234,16.13
1,9,14.80


**Pergunta 4.2:** o fim de semana e o tamanho do grupo influenciam o percentual da gorjeta da mesma forma? Justifique com os números.

**Resposta:** O fim de semana não impacta muito, pois a diferença é de apenas 0.5. Enquanto isso, o tamanho do grupo (>4 pessoas) impacta, com diferença de quase 2.



## 5. Faixa: transformar um número em categorias ordenadas
**Exemplo:** faixas de valor da conta com cortes fixos (até 15, de 15 a 25, acima de 25 dólares).

In [12]:
df["faixa_conta"] = pd.cut(df["conta"], bins=[0, 15, 25, np.inf], labels=["baixa", "média", "alta"])
df.groupby("faixa_conta", observed=True)["percentual_gorjeta"].agg(["count", "mean"]).round(2)

,count,mean
faixa_conta,,
baixa,79,18.39
média,109,15.88
alta,55,13.17


**Sua vez 5.1:** crie o atributo `faixa_conta_igual` usando `pd.qcut`, que divide as mesas em três grupos com a mesma quantidade de mesas. Mostre o menor e o maior valor de conta de cada faixa e o percentual médio da gorjeta.

In [15]:
df["faixa_conta_igual"] = pd.qcut(df["conta"],q=3,labels=["baixa", "média", "alta"])
df.groupby("faixa_conta_igual", observed=True)["percentual_gorjeta"].agg(["count", "mean"]).round(2)

,count,mean
faixa_conta_igual,,
baixa,81,18.27
média,81,16.17
alta,81,13.81


**Pergunta 5.2:** qual das duas formas de criar faixas você apresentaria ao dono do restaurante? Por quê?

**Resposta:**



## 6. Agregação: comparar cada mesa com o seu grupo
**Exemplo:** percentual médio da gorjeta do dia de cada mesa e indicador de mesa acima da média do seu dia.

In [ ]:
df["media_do_dia"] = df.groupby("dia")["percentual_gorjeta"].transform("mean")
df["acima_media_dia"] = (df["percentual_gorjeta"] > df["media_do_dia"]).astype(int)
df[["dia", "percentual_gorjeta", "media_do_dia", "acima_media_dia"]].head().round(2)

**Sua vez 6.1:** crie os atributos `media_do_turno` e `acima_media_turno`, seguindo o exemplo. Em seguida, conte quantas mesas de cada turno ficaram acima da média do próprio turno.

## 7. Os novos atributos e as aulas anteriores
**Exemplo:** padronização do percentual da gorjeta para identificar mesas fora do padrão.

In [ ]:
df["percentual_z"] = (df["percentual_gorjeta"] - df["percentual_gorjeta"].mean()) / df["percentual_gorjeta"].std()
df.nlargest(3, "percentual_z")[["conta", "gorjeta", "pessoas", "percentual_gorjeta", "percentual_z"]].round(2)

**Sua vez 7.1:** calcule a correlação de `conta`, `pessoas` e `conta_por_pessoa` com a coluna `gorjeta` e, separadamente, com o atributo `percentual_gorjeta`.

**Pergunta 7.2:** a conta se relaciona com a gorjeta e com o percentual da gorjeta da mesma forma? O que isso significa para o dono do restaurante?

**Resposta:**



## 8. Atributo próprio
**Sua vez 8.1:** crie um atributo novo que não tenha sido feito nesta aula. Escreva, na célula de resposta abaixo do código, a pergunta do dono do restaurante que ele responde e qual dos tipos vistos ele usa (razão, soma, indicador, faixa ou agregação).

**Resposta:**



**Pergunta 8.2:** com base nos atributos criados, escreva uma recomendação de até três linhas para o dono do restaurante.

**Resposta:**

